# Lang-Bridge bench — GPU speed (E1) and video identity (E2)

Measures, before we build (plan: `docs/SCALE_PLAN.md`, Phase A):

- **E1**: how fast separation, Whisper and OmniVoice run with each setting, how full the
  GPU gets, and whether faster settings change the result; separation from an Opus copy.
- **E2**: whether the same video is recognised across copies, re-downloads, formats,
  trims and added intros, and never confused with other episodes.

Works on **Colab** (Runtime → Change runtime type → T4 GPU) and **Kaggle** (File → Import
notebook → this file's GitHub link; Settings → Accelerator **GPU T4 x2**, Internet **on**).
Then **Run all** (about an hour on a T4). Paste the last cell's output back.

In [ ]:
#@title Settings
SOURCES = "https://www.youtube.com/watch?v=xwseWCSXD3Y https://www.youtube.com/watch?v=SC_opiKLohg https://www.youtube.com/watch?v=D9jZMLm72a8"  #@param {type:"string"}
#@markdown E1 videos: links or file paths, separated by spaces (add a sermon from your Drive if you like).
SECONDS = 120  #@param {type:"integer"}
#@markdown Length of each excerpt for E1.
E2_MAIN = "https://www.youtube.com/watch?v=xwseWCSXD3Y https://www.youtube.com/watch?v=SC_opiKLohg"  #@param {type:"string"}
E2_OTHERS = "https://www.youtube.com/watch?v=MSJMJxd1udk https://www.youtube.com/watch?v=hb1CBEENiPQ https://www.youtube.com/watch?v=vxoPApiNZBU https://www.youtube.com/watch?v=-idY8F7LOSE https://www.youtube.com/watch?v=m7IlyBEyi3c"  #@param {type:"string"}
#@markdown E2: videos to recognise, and other videos that must never match them.
RUN_SEPARATION = True  #@param {type:"boolean"}
RUN_WHISPER = True  #@param {type:"boolean"}
RUN_OMNIVOICE = True  #@param {type:"boolean"}
RUN_IDENTITY = True  #@param {type:"boolean"}
import os
KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))  # (Colab also has a /kaggle folder)
OUT = "/kaggle/working/lb-bench" if KAGGLE else "/content/lb-bench"
#@markdown Results go to `OUT` (`/content/lb-bench` on Colab, `/kaggle/working/lb-bench` on Kaggle).

In [ ]:
#@title Get Lang-Bridge (branch exp/scale)
import os, subprocess, sys
CODE = '/kaggle/working/lang_recomp' if KAGGLE else '/content/lang_recomp'
if not os.path.exists(CODE):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', 'exp/scale', 'https://github.com/LetPastorSpeakInYourLanguage/lang_recomp', CODE], check=True)
else:
    subprocess.run(['git', '-C', CODE, 'pull', '-q'], check=False)
sys.path[:0] = [CODE, CODE + '/worker']
os.environ.setdefault('LB_CACHE', OUT + '/cache')
print(subprocess.run(['git', '-C', CODE, 'log', '-1', '--format=Lang-Bridge %h (%cd)'], capture_output=True, text=True).stdout)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

In [ ]:
#@title E1 · separation settings and the Opus copy
from pathlib import Path
from lb_worker.bench import gpu
out = Path(OUT)
items = gpu.prepare(SOURCES.split(), out, seconds=SECONDS)
if RUN_SEPARATION:
    sep = gpu.bench_separation(items, out)
    copy = gpu.bench_audio_copy(items, out, sep)

In [ ]:
#@title E1 · Whisper batch size
if RUN_WHISPER:
    whisper = gpu.bench_whisper(items, out)

In [ ]:
#@title E1 · OmniVoice takes per call
import json
if RUN_OMNIVOICE:
    sep_dir = Path(next(r['dir'] for r in json.loads((out / 'e1_separation.json').read_text())['rows'] if r.get('dir')))
    ref, ref_text = gpu.reference(items, sep_dir, out)
    omni = gpu.bench_omnivoice(ref, ref_text, out)

In [ ]:
#@title E2 · video identity
from lb_worker.bench import identity
from lb_worker.deps import ensure
from lb_worker.stages.bulk import fetch
if RUN_IDENTITY:
    ensure('yt-dlp', probe='yt_dlp')
    d = out / 'identity'
    get = lambda url, name, h: fetch({'id': name, 'url': url}, d / f'{name}_{h}', h)
    others = [get(u, f'o{k}', 360) for k, u in enumerate(E2_OTHERS.split())]
    mains = []
    for k, u in enumerate(E2_MAIN.split()):
        vid = u.split('v=')[-1][:11]
        mains.append({'name': f'm{k}', 'video': get(u, f'm{k}', 720), 'redownload': get(u, f'm{k}', 360),
                      'other': others[0], 'links': identity.link_forms(vid) if 'youtube' in u else []})
    ident = identity.run(mains, others, out)

In [ ]:
#@title Summary (paste this back)
import json
summary = {}
for name in ('e1_separation', 'e1_audio_copy', 'e1_whisper', 'e1_omnivoice', 'e2_identity'):
    f = out / f'{name}.json'
    if f.exists():
        r = json.loads(f.read_text())
        rows = [{k: v for k, v in x.items() if k != 'dir'} for x in r.get('rows', [])]
        summary[name] = {k: v for k, v in r.items() if k not in ('rows', 'links')} | {'rows': rows}
print(json.dumps(summary, ensure_ascii=False))